## Import all the necessary libraries and restaurant review data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
reviews = pd.read_csv('data/restaurant_reviews_az.csv')

## Remove 3 star reviews from the input data, create a new column - Sentiment for the remaining reviews. For reviews with 1 or 2 star rating, set the value in the Sentiment column to 0. For reviews with 4 or 5 star rating, set the value in the sentiment column to 1.

In [ ]:
# Remove 3-star reviews
reviews = reviews[reviews['stars'] != 3].copy()

# Create Sentiment column
# 1-2 stars = 0 (Negative)
# 4-5 stars = 1 (Positive)

reviews['Sentiment'] = reviews['stars'].apply(
    lambda x: 0 if x <= 2 else 1
)

# Display the updated data
display(reviews.head())

# Check the distribution of Sentiment
print(reviews['Sentiment'].value_counts())

## Conduct necessary data processing. Prepare the training and test sets on review data for machine learning classifications. 20% of the data for testing and 80% of the data for training.

In [ ]:
# Load necessary libraries
import spacy
from sklearn.model_selection import train_test_split

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")


# Function to lowercase all text
def preprocess_text(text):

    doc = nlp(text.lower())

    # Lemmatize words and remove stop words, punctuation, and spaces
    processed_text = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

    return " ".join(processed_text)


# Apply preprocessing to review text
reviews['Processed_Text'] = reviews['text'].apply(preprocess_text)

# Display original and processed reviews
display(reviews[['text', 'Processed_Text', 'Sentiment']].head())

In [ ]:
# Define X (review text) and y (sentiment)
X = reviews['Processed_Text']
y = reviews['Sentiment']

# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=5
)

# Display the size of each dataset
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining sentiment distribution:")
print(y_train.value_counts())

print("\nTesting sentiment distribution:")
print(y_test.value_counts())

## Use Count Vectorizer and frequency count to represent documents, set maximum feature size as 1000


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer

# Tokenizer that keeps alphabetic words only
token = RegexpTokenizer(r'[a-zA-Z]+')

# Count Vectorizer using frequency counts
cv = CountVectorizer(
    stop_words='english',
    ngram_range=(1,1),
    tokenizer=token.tokenize,
    max_features=1000
)

# Fit on training data
X_train_count = cv.fit_transform(X_train)

# Transform test data
X_test_count = cv.transform(X_test)

# Display feature names
print(cv.get_feature_names_out())

In [ ]:
df = pd.DataFrame(
    X_train_count.toarray(),
    columns=cv.get_feature_names_out()
)

display(df.head())

In [ ]:
cv.vocabulary_

## Train a naive bayes classification model to classify the review sentiment and evaluate its performance.

In [ ]:
# Import Naive Bayes model and evaluation metrics
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

# Create the Naive Bayes model
mnb = MultinomialNB()

# Train the model
mnb.fit(X_train_count, y_train)

# Make predictions on the test data
y_pred = mnb.predict(X_test_count)

# Evaluate model performance
print(metrics.classification_report(y_test, y_pred))

#### Check for Overfitting

In [ ]:
# Predictions on training data
y_train_pred = mnb.predict(X_train_count)

print("Training Performance:")
print(metrics.classification_report(
    y_train,
    y_train_pred,
    target_names=['Negative', 'Positive']
))

print("Testing Performance:")
print(metrics.classification_report(
    y_test,
    y_pred,
    target_names=['Negative', 'Positive']
))

#### The Multinomial Naive Bayes model performed well in classifying restaurant review sentiment, achieving an overall accuracy of 92%. The model performed particularly well at identifying positive reviews, with 94% precision, 95% recall, and an F1-score of 0.94. Performance on negative reviews was slightly lower, with 87% precision, 85% recall, and an F1-score of 0.86. This difference may partially reflect the class imbalance in the dataset, as positive reviews represent a larger portion of the test data. There is also little evidence of overfitting as the positive and negative rewviews only marginally improve as well as the overall accuracy.

## Train a SVM model to classify the review sentiment and evaluate its performance

In [ ]:
# Import SVM
from sklearn import svm

# Create the SVM model
svm_model = svm.SVC()

# Train the model
svm_model.fit(X_train_count, y_train)

In [ ]:
# Make predictions on the test data
svm_pred = svm_model.predict(X_test_count)

# Evaluate model performance
print("Testing Performance:")
print(metrics.classification_report(
    y_test,
    svm_pred,
    target_names=['Negative', 'Positive']
))

#### Check for overfitting

In [ ]:
# Make predictions on training data
svm_train_pred = svm_model.predict(X_train_count)

print("Training Performance:")
print(metrics.classification_report(
    y_train,
    svm_train_pred,
    target_names=['Negative', 'Positive']
))

print("Testing Performance:")
print(metrics.classification_report(
    y_test,
    svm_pred,
    target_names=['Negative', 'Positive']
))

#### The SVM model demonstrated strong performance in classifying restaurant review sentiment, achieving 94% accuracy on the test data. For negative reviews, the model achieved 90% precision, 88% recall, and an F1-score of 0.89. For positive reviews, it achieved 95% precision, 96% recall, and an F1-score of 0.96. The SVM also slightly outperformed the Naive Bayes model particularly for the negative reviews. Training accuracy for the SVM was 97% compared with 94% testing accuracy, indicating some modest overfitting, however, the relatively small difference and strong test performance suggest that the model still generalizes well to unseen reviews.

## Use TF-IDF vectorizer to represent the documents and set the max feature size to 1000

In [ ]:
# Import TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import RegexpTokenizer

# Tokenizer that keeps alphabetic words only
token = RegexpTokenizer(r'[a-zA-Z]+')

# Create TF-IDF Vectorizer with maximum 1000 features
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,1),
    tokenizer=token.tokenize,
    max_features=1000
)

# Fit vectorizer on training data and transform
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform testing data using the same vectorizer
X_test_tfidf = vectorizer.transform(X_test)

# Display the dimensions
print("Training Data Shape:", X_train_tfidf.shape)
print("Testing Data Shape:", X_test_tfidf.shape)

# Display the features
print(vectorizer.get_feature_names_out())

In [ ]:
df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

display(df.head())

##  Train a naive bayes classification model with TF-IDF feature values to classify the review sentiment and evaluate its performance

In [ ]:
# Create the Naive Bayes model
mnb_tfidf = MultinomialNB()

# Train the model using TF-IDF features
mnb_tfidf.fit(X_train_tfidf, y_train)

# Make predictions on the test data
tfidf_pred = mnb_tfidf.predict(X_test_tfidf)

# Evaluate model performance
print("Testing Performance:")
print(metrics.classification_report(
    y_test,
    tfidf_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
# Make predictions on training data
tfidf_train_pred = mnb_tfidf.predict(X_train_tfidf)

print("Training Performance:")
print(metrics.classification_report(
    y_train,
    tfidf_train_pred,
    target_names=['Negative', 'Positive']
))

#### The model performed especially well on positive reviews, achieving 97% recall and an F1-score of 0.94. Performance for negative reviews was somewhat weaker, with 91% precision, 74% recall, and an F1-score of 0.82. This indicates that while the model is highly accurate when predicting a review as negative, it still misclassifies some actual negative reviews as positive. Compared with the frequency-count Naive Bayes model, TF-IDF produced a about the same in overall accuracy, although the frequency-count model provided somewhat better recall for negative reviews. Overall, TF-IDF provides strong sentiment classification performance but shows some bias toward correctly identifying the more common positive class. There is also no strong evidenve of overfitting.

##Train a SVM model with TF-IDF feature value to classify the review sentiment and evaluate its performance

In [ ]:
# Create the SVM model
svm_tfidf = svm.SVC()

# Train the model using TF-IDF features
svm_tfidf.fit(X_train_tfidf, y_train)

In [ ]:
# Make predictions on the test data
svm_tfidf_pred = svm_tfidf.predict(X_test_tfidf)

# Evaluate model performance
print("Testing Performance:")
print(metrics.classification_report(
    y_test,
    svm_tfidf_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
# Make predictions on training data
svm_tfidf_train_pred = svm_tfidf.predict(X_train_tfidf)

print("Training Performance:")
print(metrics.classification_report(
    y_train,
    svm_tfidf_train_pred,
    target_names=['Negative', 'Positive']
))

#### The SVM model using TF-IDF features achieved strong classification performance, with an overall test accuracy of 95%. For negative reviews, the model achieved 91% precision, 88% recall, and an F1-score of 0.90. For positive reviews, it achieved 96% precision, 97% recall, and an F1-score of 0.96. These results represent the strongest test performance of the models evaluated so far. The model achieved 99% accuracy on the training data compared with 95% on the testing data, suggesting some evidence of overfitting. However, the model still generalizes very well to unseen reviews. Overall, SVM with TF-IDF provides the strongest sentiment classification performance among the feature and model combinations tested.

##  Use VaderSentiment to predict the review sentiment and evaluate its performance

In [ ]:
!pip install -q vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Create VADER sentiment analyzer
sentiment = SentimentIntensityAnalyzer()

In [ ]:
# Predict sentiment using VADER
vader_pred = []

for text in X_test:

    sent = sentiment.polarity_scores(text)

    if sent['compound'] > 0:
        vader_pred.append(1)
    else:
        vader_pred.append(0)

In [ ]:
# Evaluate VADER performance
print("VADER Performance:")
print(metrics.classification_report(
    y_test,
    vader_pred,
    target_names=['Negative', 'Positive']
))

####VADER achieved an overall accuracy of 83% when classifying restaurant review sentiment. Performance differed substantially between the two sentiment classes. For positive reviews, VADER achieved 82% precision, 98% recall, and an F1-score of 0.90, indicating that it successfully identified nearly all positive reviews. However, performance was considerably weaker for negative reviews. Although negative precision was high at 90%, recall was only 44%, resulting in an F1-score of 0.59. This indicates that VADER frequently misclassified actual negative reviews as positive. The macro-average F1-score of 0.74 further demonstrates the imbalance in performance between the two classes. Overall, VADER was less effective and less balanced than the supervised machine-learning models, particularly for identifying negative restaurant reviews.

## Compare the model performances and write down your observations on the performance of lexicon-based sentiment analysis approach and different machine learning models

#### Overall, the supervised machine learning models outperformed VADER in classifying restaurant review sentiment. Using frequency counts, the Naive Bayes model achieved approximately 92% test accuracy, while the SVM improved performance to approximately 94%. When TF-IDF features were used, Naive Bayes achieved approximately 91% accuracy, while SVM produced the strongest overall performance at approximately 95% accuracy, although its higher training accuracy of 99% suggests some modest overfitting. VADER produced the lowest overall accuracy at 83% and struggled particularly with negative reviews, achieving only 44% recall despite 90% precision. Across the models, positive reviews were generally easier to classify than negative reviews, which may be influenced by the larger number of positive reviews in the dataset. Overall, SVM with TF-IDF provided the strongest and most balanced classification performance, demonstrating that both the choice of classification algorithm and the method used to represent text can have a meaningful impact on sentiment analysis results.


## Acknowledge if you have used any GenAI tools in this assignment and anyone you have worked together with on this assignment.

#### I worked on this assigment by myself. LLM was used to debug code and verify validity of outputs.

## Render the HTLM file of your Python Notebook with the following commands at the end of your notebook

In [ ]:
# HTML Rendering
!pip install jupyter
!pip install nbconvert
!jupyter nbconvert "data/LA2_Danut_George.ipynb" --to html